# BME 590 - Workshop 4 - Modular Cloning
**Professor:** Emma Chory, Ph.D.

**Authors:** 
Rick Wierenga, Joe Laforet, Stefan Golas, Ben Perry

---

### Usage Note
**Reminder** - You should be running this notebook **locally** on **VS Code** not navigating it through **GitHub**.

**Reminder 2** - Open this workshop with `uv run bme590 start 04`. Every `uv run bme590` command pulls the latest class materials first, so you get the most recent fixes (`uv run bme590 update` does only that). See [Get started in the class README](https://github.com/chory-lab/bme590-fall-2026#get-started)

---

---

#### ⚠️ Check your kernel first (top-right corner)

It should read **BME 590 (lab automation)**. If it says *“Select Kernel”*, a Python version, or anything else:

1. Click the kernel picker in the top-right.
2. If VS Code offers **Select Another Kernel…**, click it first. Then choose **Python Environments…** — *not* **Existing Jupyter Server…**, which asks for a URL like `127.0.0.1` and is a dead end.
3. Pick the class environment: the `.venv` inside the class folder. Depending on your VS Code version it is listed as **BME 590 (lab automation)** or as `.venv (Python 3.11)`. (**Jupyter Kernel… → BME 590 (lab automation)** reaches the same environment.)

Wrong kernel = `ModuleNotFoundError` / *"requires the ipykernel package"* on the very first cell. The fix is the 10 seconds above — **do not pip-install anything**.

---


## Working from Files

In lab automation, it can be useful to load instructions or "work orders" from a file. 

For example, imagine a pipeline that takes a traditional bioinformatic analysis pipeline to determine which genes are most expressed in a given experiment, and subsequently uses **mapped locations** of the relevant wells on a liquid handling deck to proceed with an experiment. 

Being able to not only connect an outside workflow to the robot via file transfer, but also being able to **map biological elements** to their respective wells and plates on a deck can help integrate workflows even further.

Another example is Golden Mutagenesis, a technique that uses the modularity of golden gate cloning to generate diverse plasmid libraries. Libraries of DNA fragments can be assembled as modules to generate combinatorial diversity from a relatively small number of individual parts that can be assembled in a standardized way.

This technique [has been used](https://www.nature.com/articles/s41598-019-47376-1) to evolve a number of enzymes in recent years.

### Pandas, Polars, and Golden Gate Assembly

One of the most common types of data file is the **.csv** file, or comma separated values file. There are, of course, nearly infinite ways to store information in file formats; however, let's suppose we have a **mapping** of DNA fragment IDs and the wells which each fragment is in on the liquid handling deck.

To illustrate this scenario, let's set up our PLR deck. First, imports:

In [ ]:
# standard imports
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import LiquidHandlerChatterboxBackend
from pylabrobot.resources.opentrons import OTDeck

# resources for deck setup
from pylabrobot.resources import (
    Deck,
    set_tip_tracking,
    set_volume_tracking,
    cor_96_wellplate_360uL_Fb,
    opentrons_96_tiprack_1000ul,
)

# import os
import os

Enable error tracking for cross-contamination, volumes, and tips.

In [ ]:
set_tip_tracking(enabled = True)
set_volume_tracking(enabled = True)


Now import the course's standard deck visualization helpers

In [ ]:
# Course-standard deck setup: `visualize_deck` builds the LiquidHandler + visualizer
# pair and attaches the visualizer to the handler as `lh.vis`, so exercise code
# below can record GIFs with `gif_recorder(lh.vis, name="exercise_N.gif")`.
# Everything between `await rec.start()` and `await rec.stop()` is recorded; the
# GIF downloads automatically when recording stops.
from bme590.visualizer_ext import gif_recorder, visualize_deck, close_visualizer

Now, let's set up our OpenTrons-2 deck with plates for our DNA fragments and a plate to combine them all together on.

In [ ]:
async def make_golden_gate_ot2():

    # instantiate deck
    deck = OTDeck()

    # add plates with fragments
    plate_slots = [4, 5, 6]
    for i, plate_slot in enumerate(plate_slots):
        deck.assign_child_at_slot(cor_96_wellplate_360uL_Fb(name = f"fragments_{i}"), plate_slot)
    
    # add tip racks
    tip_rack_slots = [1, 2, 3]
    for i, tip_slot in enumerate(tip_rack_slots):
        deck.assign_child_at_slot(opentrons_96_tiprack_1000ul(name = f"tip_rack_{i}"), tip_slot)
    
    # add working plate
    deck.assign_child_at_slot(cor_96_wellplate_360uL_Fb(name = f"cloning_plate"), 10)

    return deck

# call function
deck = await make_golden_gate_ot2()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Now, assume we have created a mapping file of DNA ID - Plate Name - Well Location and saved it as a `.csv` file. We need some way to **load the file into memory** to see our ID mapping.

Luckily, one of the most widely used Python libraries is Pandas, which enables you to load data into a **table in memory, known as a DataFrame**, upon which you can do operations. Pandas is already installed in the class environment.

To start, let's **import pandas**

In [ ]:
import pandas as pd
print(pd.__version__)

If this cell fails with `ModuleNotFoundError: No module named 'pandas'`, your notebook is not on the class kernel. Go back to the kernel check at the top of this notebook. **Do not install pandas yourself** - it is already part of the class environment.

Now let's **load our CSV** into memory using the `read_csv()` function. The data lives in the `data/` folder of the class repo.

---

**Note:** If you have not used pandas before, you may find [this](https://pandas.pydata.org/docs/user_guide/10min.html) tutorial helpful. **You WILL be using Pandas in the exercises for this workshop, so this is recommended reading!**

---

In [ ]:
# This notebook runs from the class folder's assignments/ directory, so the
# class folder itself -- the one holding data/ -- is its parent.
class_dir = os.path.dirname(os.getcwd())
print(class_dir)

**Important:** You are telling Python where to look for the data you will load. The path printed above should be your class folder - it ends in `bme590-fall-2026` and contains the `data/` folder. If it does not, you are probably running a copy of this notebook from outside `assignments/`: open it with `uv run bme590 start 04` instead. As a last resort, set `class_dir` in the cell above to the full path of your class folder, as a string.

In [ ]:
# define the path pointing to our data file
csv_path = os.path.join(class_dir, "data", "fragments.csv")

# read the dataframe to RAM
df = pd.read_csv(csv_path)

# print the first 5 rows of the data
df.head() # <- NOTE: df.head(n) prints the first n rows of a dataframe. df.tail(n) does the same but for the last n rows.

As you can see, we have the following columns:

- `well_id` - The identifier of the well for each DNA fragment

- `fragment_id` - The identifier of the DNA fragment itself

- `plate_id` - The identifier of the plate itself.

- `volume (uL)` - The volume of that fragment in a given plate well.

We can now write a for loop to iterate through each row using `df.iterrows()`. **Note** that this function returns a tuple at each iteration of the form `index, row`, where `row` is a pandas **Series**: it works like a dictionary of column_name-value pairs, so `row['well_id']` gets that row's well.

The below code will run a for loop and show you how to iterate through the loaded dataframe and extract the values for each column.

**Note -** The `break` keyword exits the loop immediately, so only one iteration is done, as an example print statement.

In [ ]:
for idx, row in df.iterrows(): # iterate over rows
    print(f"{row.keys()=}") # print the keys in the row dictionary
    print()
    print(f"{row['well_id']=}") # extract the well id
    print(f"{row['fragment_id']=}") # extract the fragment id
    print(f"{row['plate_id']=}") # extract the plate id
    print(f"{row['volume (uL)']=}") # extract the volume
    break

You can also filter the dataframe by values in certain columns. For example, filtering it so that only plate 2 is present in the data.

In [ ]:
df_plate_2 = df[df['plate_id'] == 2]
df_plate_2.head()

You can also filter by more specific things, such as getting the plate and well containing a fragment id of interest

In [ ]:
df_specific_fragment = df[df["fragment_id"] == "X095"]
df_specific_fragment.head()

Since there are multiple wells containing our fragment of interest, we can **randomly choose one** like this.

**Hint -** This is probably going to be useful in the exercises. If there were a scenario where you needed more volume than was present in a singular well, you may need to use information from multiple rows. However, for this workshop, we will keep it simple

In [ ]:
row = df_specific_fragment.sample(n = 1)
row

These are the basics for what you need to complete this workshop, but we encourage you to explore other pandas functionalities.

As a side note, if you ever work with **large scale datasets** such as those in the gigabyte range, a useful library is [Polars](https://pola.rs/), which enables large-scale multi-threaded dataframe operations, in a lower level language, and even [on the GPU](https://pola.rs/posts/gpu-engine-release/).

## Exercises

---

**TO-DO:** For each of the following exercises, since they are a bit different from each other, we will require submission of one or more of the following:

- a `.txt` file containing your code.

- a `.gif` file containing a GIF animation of your protocol running (recorded by the `gif_recorder` calls in the exercise cells)

- a `.png` or `.jpg` image of your deck setup, if needed.

- a `.csv` file, for some exercises

We will explicitly tell you for each exercise and sub-exercise, which items to submit. There will be a **sample submission format** for each exercise.

---

Some exercises below will ask you to define your own **functions or classes**. We will provide the **function or class name** and sometimes the **input argument names** for you, but in general, the body of the functions is up to you.

Once you get your protocol working as intended for each problem, you will need to **record a GIF** of your protocol running for each exercise, as directed

**IMPORTANT** - You should be judiciously commenting your code to explain its function. We will grade every problem by **quality of code**. Excessively long code or lack of comments will be subject to **point deduction**

Furthermore, you can write the code how you see fit. However, **do not change function names**. If your code uses anything that is **not already imported at the top of this notebook**, include those imports at the top of your .txt file submission.

---


### **Exercise 1.** Deck Setup (15 pts)

First, given the deck we setup, complete the following function `add_fragments()` which should take in the following parameters:

- `deck` - The already set up OT-2 deck from earlier in this workshop.

- `fragment_csv_path` - The path to the fragment data we were using earlier.

Using the techniques learned from the **Liquid handling** workshop about **setting initial liquids**, write a function that puts the correct volume of each fragment in its target well, solely determined by the data in the `fragments.csv` file.

**Note -** `plate_id` in the CSV runs **1, 2, 3**, while the deck's fragment plates are named **`fragments_0`, `fragments_1`, `fragments_2`**. Plate 1 is `fragments_0`, and so on.

**Note -** If you use `tracker.set_liquids(...)`, PyLabRobot prints a `DeprecationWarning` suggesting `set_volume`. Either works for this workshop; the warning is harmless.

In [ ]:
def add_fragments(deck : Deck,
                  fragment_csv_path : str):
    ... # YOUR CODE HERE

# call the function
add_fragments(deck, csv_path)

Submit your code as `exercise_1.txt`. Make sure it is commented. Only include imports that are not already at the top of the notebook.

---


### **Exercise 2.** Work Order Parsing (25 pts)

We have now been provided a list of work orders of Golden Mutagenesis targets in the `cloning.csv` file. Let's go ahead and take a look at some example rows of this dataframe. 

In [ ]:
# set the path
work_order_csv_path = os.path.join(class_dir, "data", "cloning.csv")

# read the dataframe to RAM
work_order_df = pd.read_csv(work_order_csv_path)

# print the first 5 rows of the data
work_order_df.head()

Great! This file contains two columns:

- `well_id` - The well ID on the `cloning_plate` of fragments to combine. **Note -** the well ID index now is of the form A01, which should translate to A1. You will have to find a way to translate these IDs. See hint below.

- `fragment_tuple` - A **string** that looks like a tuple, of form: `(ID_1,ID_2,ID_3)`, where each ID corresponds to a DNA fragment. You will need to turn it into a real tuple of three IDs (string methods like `.strip()` and `.split()` help).

For this exercise, you will need to write a **generator** called `extract_fragment_combinations()` which creates a for loop over the rows in the dataframe, and yields data in the form: `well_id, fragments` where:

- `well_id` is the **converted** id from the form "A01" to "A1" for all combinations of IDs.

- `fragments` is a **tuple** of the form **fragment_X, fragment_Y, fragment_Z**

**HOWEVER** Our list was assembled by a mad scientist! Many of the combinations - **74 of the 96** - point to at least one ID which **doesn't exist** in the fragments set at all. Those work orders cannot be filled, so the generator should **skip them**: it yields only the work orders whose three fragments all exist (22 of them).

**2.A.** Let's break down this problem piece by piece. First, let's write a function to convert the **well ID** to the right format. This should simply input the `old_well_id` and convert it to a normalized ID. To do this:

- Index the first character of the `ID` as the **row_id** for a given number.

- For the rest of the string, convert it to an **integer** using the `int()` conversion function.

- Finally, using [string concatenation](https://www.w3schools.com/python/gloss_python_string_concatenation.asp) or f-strings, combine the row_id and integer number together and return it.

In [ ]:
def convert_well_id(old_well_id: str):
    ... # YOUR CODE HERE

**2.B.** Great! Now we need a function called `get_well()` which can input:

- `fragment_id` - The identifier of the fragment we are checking is present.

- `fragment_df` - The dataframe of fragment IDs and locations, originally loaded as **df**

The function should use pandas filtering to:

- Filter down the dataframe to only rows where `fragment_id` **equals** the fragment ID being searched for.

- Implement logic that checks if the length of the filtered dataframe is 0. If so, return `None, None`

- Otherwise, sample **only one row** and use the `.item()` function to get the specific cell entry for given row and return a tuple containing data of the form `well_id, plate_name`, where `plate_name` is the **deck** name of the plate (e.g. `"fragments_0"`), not the number from the CSV

In [ ]:
def get_well(fragment_id: str,
             fragment_df: pd.DataFrame):
    ... # YOUR CODE HERE

**2.C.** Finally, implement the `extract_fragment_combinations` function, which will input the work-order and existing fragments CSV path and:

- Load each dataframe into memory using `pd.read_csv(...)`

- Iterate through the rows of the work order dataframe using `iterrows()`

- For each row, extract the `fragment_tuple`, turn it into a tuple of three IDs, and implement logic to check that each fragment is valid:

    - For each fragment in the fragment tuple, call `get_well()`. If `None, None` is returned for any of them, then set the boolean flag `all_fragments_valid` to False.

- If `all_fragments_valid` is still True, **yield** data of the form `well_id, fragment_tuple`. Otherwise yield nothing for that row - it is skipped.

In [ ]:
def extract_fragment_combinations(cloning_csv_path: str | os.PathLike,
                                  fragment_csv_path: str | os.PathLike):

    ... # YOUR CODE HERE

    # iterate over work orders
    for _, row in ...:

        # boolean logic implementation
        all_fragments_valid = True 
        
        ... # YOUR CODE HERE

        # scaffold logic
        for ... in ...:
            if ...:
                all_fragments_valid = False
                break  

        # only if fragments are valid, then yield the relevant results.
        if all_fragments_valid:
            well_id = convert_well_id(...)
            yield well_id, fragment_tuple

Great! Submit your code as a file named `exercise_2.txt`. Make sure it is commented. Only include imports that are not already at the top of the notebook.

---

### **Exercise 3.** Modular Cloning (40 pts)

Now that we have the deck setup and a way to iterate over our work-order data, we should now design a liquid handling protocol that will achieve the following:

- Iterate over the work order list to get the well for cloning and the three fragments to pipette together.

- For each well in this data, do the following:

    - For each DNA fragment in the fragments to combine, find the well and plate it is located in from the `fragments.csv`.

    - Using this information, pipette **30 uL of each fragment** to the target well in the work order list. **You should use a separate tip for each fragment.**

    - Finally, **mix** the **90 uL** contents in the target well together and move on to the next well

Since you have now gained experience in writing liquid handling protocols, we will not provide any **strict guidelines on helper functions** or structure of the protocol for this exercise. 

However, do note that your code will still be graded for its **efficiency** and **quality of comments**. At a minimum, you should aim to have (and we will look for):

- A function to perform a mixing operation.

- A function to perform one entire pipetting step given a well ID and a triplet to combine.

- A function to find the appropriate well given a fragment ID.

For each function you write, make sure to include **robust comments** as to how it works. It may be helpful to look back at **workshop 2** for inspiration of function structure.

Your final protocol should run through the `run_protocol_exercise_3()` function.

Comment your code well and include all imports **not included at the top of the notebook** that you may need. 

**Include a description of each helper function in the commented section below.**

In [ ]:
# add any imports needed that are not already at the top of the notebook

# --- HELPER FUNCTIONS ---

# Function N Code:
# Function N Description: 

# Function N Code:
# Function N Description: 

# Function N Code:
# Function N Description: 

async def run_protocol_exercise_3(deck: Deck,
                                  lh: LiquidHandler,
                                  fragment_df_path: str | os.PathLike):

    ... # YOUR CODE HERE

    # for each work order whose fragments all exist, assemble it. The generator
    # from exercise 2 already skips the orders that name a missing fragment.
    for well_id, fragments in extract_fragment_combinations(work_order_csv_path, fragment_df_path):
        ...

# setup the deck
deck = await make_golden_gate_ot2()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

# add fragments
add_fragments(deck, csv_path)

# run protocol; everything between rec.start()/rec.stop() is recorded to
# exercise_3.gif, which downloads automatically when you stop.
rec = gif_recorder(lh.vis, name="exercise_3.gif")
await rec.start()
await run_protocol_exercise_3(deck, lh, csv_path)
await rec.stop()

Submit a GIF of your protocol running as `exercise_3.gif` and a code file of your code **and function explanations** as `exercise_3.txt`

---

### **Exercise 4.** State Verification (15 pts)

Now that you have completed your experiment, you should utilize volume tracking to write a function `verify_results()` which is able to iterate over all the original work orders and verify, for each order, whether or not it was completed successfully.

To do this, simply:

- Iterate over **every** work order in `cloning.csv` (not just the valid ones the generator yields).

- Look up the order's well on the `cloning_plate` and read how much liquid it holds with `well.tracker.get_used_volume()`. PyLabRobot 0.2.x does not track *which* liquid a transfer moved -- only total volume -- so volume is the only thing you can check:

    - A well holding **0 uL** was never touched: the order was skipped because one of its fragments is missing.

    - Any other well should hold the full **90 uL** (30 uL of each of the 3 fragments). Assert that it does.

- For every **skipped** order, add the well_id to the `empty_wells` list and that order's **three** fragment IDs (all three, not only the missing one) to `fragment_x_list`, `fragment_y_list`, and `fragment_z_list`.

- The cell below saves your results to an output `exercise_4.csv` file in the `data/` folder. Submit this along with the code for exercise 4.

In [ ]:
def verify_results(cloning_csv_path : str,
                   deck: Deck):

    ... # YOUR CODE HERE

    # lists to store empty wells and fragment tuples for saving as a DataFrame
    empty_wells = []
    fragment_x_list = []
    fragment_y_list = []
    fragment_z_list = []

    # for each work order
    for ... in ...:

        # how much liquid the order's well on the cloning plate holds
        used_volume = ...

        # an untouched well means the order was skipped
        if used_volume == 0:
            empty_wells.append(...)
            fragment_x_list.append(...)
            fragment_y_list.append(...)
            fragment_z_list.append(...)

        # otherwise, verify the order was filled completely. PyLabRobot 0.2.x
        # does not remember which liquid moved across a transfer, so compare
        # volumes: each valid order dispenses 30 uL of each of the 3 fragments.
        else:
            expected = ...
            assert ..., f"{well_id} was not filled completely!"

    # convert results to a dataframe
    result = pd.DataFrame(
        {
            "well_id" : empty_wells,
            "fragment_x" : fragment_x_list,
            "fragment_y" : fragment_y_list,
            "fragment_z" : fragment_z_list,
        }
    )

    return result

# save data to file
save_path = os.path.join(class_dir, "data", "exercise_4.csv")
result_df = verify_results(work_order_csv_path, deck)
result_df.to_csv(save_path, index = False)

Submit the resulting data file as `exercise_4.csv` and the code as `exercise_4.txt`.


---

#### Conclusion

That's all for workshop 4! Double check that you have submitted a `.txt` file for every exercise, a `.gif` file for exercise 3 and a `.csv` file for exercise 4. You should have submitted:

- `exercise_1.txt` with the code for exercise 1.

- `exercise_2.txt` with the code for exercise 2.

- `exercise_3.txt` with the code and function descriptions for exercise 3.

- `exercise_3.gif` showing the full protocol in exercise 3, starting with the set up OT-2 deck with DNA fragments.

- `exercise_4.txt` with the code for exercise 4.

- `exercise_4.csv` with the resulting data from running your code in exercise 4.

**EVERY CODE BLOCK SHOULD HAVE WELL-WRITTEN COMMENTS**

If you are still feeling unsure on deck setup, please **reach out to the teaching team**, contact info for whom can be found in the `README.md` file on the [class GitHub](https://github.com/chory-lab/bme590-fall-2026)

---

---

#### Finished: close the visualizer

`visualize_deck` closes the previous visualizer each time you call it, so you only ever have one running. The last one is still up, though -- it holds ports 1337 and 2121 until this kernel dies, and a later workshop opened while it is still running will quietly land on the next port up and drive a page you are not looking at.

Run this before you close the notebook.


In [ ]:
await close_visualizer()
